### THIS IS THE PROJECT OF NEW YORK TAXI FARE PREDICTION


In [ ]:
import pandas as pd
import opendatasets as od
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor       
from sklearn.ensemble import RandomForestRegressor   
from xgboost import XGBRegressor                     
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/dansbecker/new-york-city-taxi-fare-prediction")

In [ ]:
 
df = pd.read_csv('train.csv',nrows=104876)

In [ ]:
import os
file_size = os.path.getsize('new-york-city-taxi-fare-prediction/train.csv') / (1024**3)   # GB mein
print(f"File size: {file_size:.2f} GB")

In [ ]:
df.info()

In [ ]:
 
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Ab usse naye, useful features nikaalo
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek  
df['month'] = df['pickup_datetime'].dt.month
df['year'] = df['pickup_datetime'].dt.year
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

 df = df.drop('pickup_datetime', axis=1)

In [ ]:
df =df.drop("key",axis=1)

In [ ]:
 
def calculate_distance(lat1, lon1, lat2, lon2):
     lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c   
    return km

df['distance_km'] = calculate_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

 df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 200)]

 df = df[(df['distance_km'] > 0) & (df['distance_km'] < 100)]


import numpy as np

def calculate_distance(lat1, lon1, lat2, lon2):
     lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c
    return km
 
jfk_lat, jfk_lon = 40.6413, -73.7781         
lga_lat, lga_lon = 40.7769, -73.8740        
ewr_lat, ewr_lon = 40.6895, -74.1745       

 
df['pickup_dist_jfk'] = calculate_distance(df['pickup_latitude'], df['pickup_longitude'], jfk_lat, jfk_lon)
df['pickup_dist_lga'] = calculate_distance(df['pickup_latitude'], df['pickup_longitude'], lga_lat, lga_lon)
df['pickup_dist_ewr'] = calculate_distance(df['pickup_latitude'], df['pickup_longitude'], ewr_lat, ewr_lon)

 
df['dropoff_dist_jfk'] = calculate_distance(df['dropoff_latitude'], df['dropoff_longitude'], jfk_lat, jfk_lon)
df['dropoff_dist_lga'] = calculate_distance(df['dropoff_latitude'], df['dropoff_longitude'], lga_lat, lga_lon)
df['dropoff_dist_ewr'] = calculate_distance(df['dropoff_latitude'], df['dropoff_longitude'], ewr_lat, ewr_lon)

 
df['is_airport_trip'] = (
    (df['pickup_dist_jfk'] < 2) | (df['dropoff_dist_jfk'] < 2) |
    (df['pickup_dist_lga'] < 2) | (df['dropoff_dist_lga'] < 2) |
    (df['pickup_dist_ewr'] < 2) | (df['dropoff_dist_ewr'] < 2)
).astype(int)

print(df[['pickup_dist_jfk', 'dropoff_dist_jfk', 'is_airport_trip']].head())

In [ ]:

X = df.drop('fare_amount', axis=1)    
y = df['fare_amount']                 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [ ]:
model = LinearRegression(n_jobs=-1)
model.fit(X_train,y_train)

In [ ]:
model.predict(X_test)

In [23]:
models = {
      'XGBoost':XGBRegressor(
    n_estimators=500,
     n_jobs=-1,
    max_depth=8,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

}
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{name} -> RMSE: {rmse:.3f}")

XGBoost -> RMSE: 3.565


In [24]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")

Train RMSE: 2.692
Test RMSE: 3.565
